In [1]:
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import librosa
import math
import time

In [22]:
from google.colab import userdata
userdata.get('secretName')

'secretN'

In [23]:
#Load the api client id and secret from file
f = open('apikeys.json')
apikeys = json.load(f)
CLIENT_ID = apikeys['clientId']
CLIENT_SECRET = apikeys['clientSecret']

In [24]:
#get access token
def authenticate_token():
    AUTH_URL = 'https://accounts.spotify.com/api/token'

    auth_response = requests.post(AUTH_URL, {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
    })

    # convert the response to JSON
    auth_response_data = auth_response.json()

    # save the access token
    access_token = auth_response_data['access_token']

    headers = {
        'Authorization': f'Bearer {access_token}'
    }
    return headers

headers = authenticate_token()

In [25]:
# base URL of all Spotify API endpoints
BASE_URL = 'https://api.spotify.com/v1/'

genre_seeds = requests.get(BASE_URL + 'recommendations/available-genre-seeds', headers=headers)

In [7]:
# genre_seeds = genre_seeds.json()['genres']

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [26]:
import requests, time

BASE_URL = "https://api.spotify.com/v1"

def get_headers():
    # your authenticate_token() from above
    return authenticate_token()

def search_artists_by_genre(genre="pop", max_artists=500, page_size=50):
    """
    Returns up to `max_artists` artists tagged with the given genre.
    Note: Spotify's genre tags exist mostly for well-known artists; results aren't exhaustive.
    """
    headers = get_headers()
    seen_ids, artists = set(), []
    offset = 0
    while len(artists) < max_artists:
        params = {
            "q": f'genre:"{genre}"',  # quotes help with multi-word genres
            "type": "artist",
            "limit": page_size,
            "offset": offset,
        }
        r = requests.get(f"{BASE_URL}/search", headers=headers, params=params)
        r.raise_for_status()
        items = r.json().get("artists", {}).get("items", [])
        if not items:
            break
        for a in items:
            if a["id"] not in seen_ids:
                seen_ids.add(a["id"])
                artists.append({"id": a["id"], "name": a["name"]})
                if len(artists) >= max_artists:
                    break
        offset += page_size
        time.sleep(0.1)  # polite pacing vs rate limits
    return artists

pop_artists = search_artists_by_genre("pop", max_artists=50)
print([a["id"] for a in pop_artists[:50]])


['1scVfBymTr3CeZ4imMj1QJ', '00x1fYSGhdqScXBRpSj3DW', '6jJ0s89eD6GaHleKKya26X', '53XhwfbYqKCa1cC15pYq2q', '0TnOYISbd1XYRBk9myaseg', '33qOK5uJ8AR2xuQQAhHump', '74XFHRwlV6OrjEM0A2NCMF', '4gzpq5DPGxSnKTe4SA8HAU', '08GQAI4eElDnROBrJRGE0X', '3AA28KZvwAUcZuOKwyblJQ', '2dIgFjalVxs4ThymZ67YCE', '6M2wZ9GZgrQXHCFfjv46we', '1QAJqy2dA3ihHBFIHRphZj', '12GqGscKJx3aE4t07u7eVZ', '7mW7Tv7NvywKKXqafZo0Lc', '6LuN9FCkKOj5PcnpouEgny', '25uiPmTg16RbhZWAqwLBy5', '0hCNtLu0JehylgoiP8L4Gh', '4tuJ0bMpJh08umKkEXKUI5', '0du5cEVh5yTK9QJze8zA0C', '1HY2Jd0NmPuamShAr6KMms', '04gDigrS5kc9YWfZHwBETP', '00FQb4jTyendYWaN8pK0wa', '4oUHIQIBe0LHzYfvXNW4QM', '5pKCCKE2ajJHZ9KAiaK11H', '1McMsnEElThX1knmY4oliG', '1Cs0zKBU1kc0i8ypK3B9ai', '74KM79TiuVKeVCqs8QtB0B', '6qqNVTkY8uBg9cP3Jd7DAH', '6vWDO969PvNqNYHIOW5v0m', '6eUKZXaKkcviH0Ku9w2n3V', '1uNFoZAHBGtllmzznpCI3s', '6pV5zH2LzjOUHaAvENdMMa', '7GlBOeep6PqTfFi59PTUUN', '31TPClRtHm23RisEBtV3X7', '66CXWjxzNUsdJxJ2JdwvnR', '06HL4z0CvFAxyc27GXpf02', '2yNNYQBChuox9A5Ka93BIn', '0RMJOzHDhA

In [27]:
import pandas as pd

pop_artists = search_artists_by_genre("pop", max_artists=200)

pop_df = (
    pd.DataFrame(pop_artists)[["name", "id"]]
      .rename(columns={"name": "artist_name", "id": "artist_id"})
      .drop_duplicates(subset=["artist_id"])
      .reset_index(drop=True)
)

pop_df.head()
# pop_df.to_csv("pop_artists.csv", index=False)  # optional


,artist_name,artist_id
0,Forrest Frank,1scVfBymTr3CeZ4imMj1QJ
1,Olivia Dean,00x1fYSGhdqScXBRpSj3DW
2,Katy Perry,6jJ0s89eD6GaHleKKya26X
3,Imagine Dragons,53XhwfbYqKCa1cC15pYq2q
4,Pitbull,0TnOYISbd1XYRBk9myaseg


In [28]:
print(pop_df.shape)

(200, 2)


In [12]:
# from pyarrow import feather
# feather.write_feather(pd.DataFrame(genre_seeds, columns=['genre']), 'data/genre_seeds.feather')

FileNotFoundError: [Errno 2] Failed to open local file 'data/genre_seeds.feather'. Detail: [errno 2] No such file or directory

In [29]:
results =[]
for idx, genre in enumerate(genre_seeds):
    params = {
        'seed_genres':genre,
        'limit':100
    }

    recs = requests.get(BASE_URL + 'recommendations', params=params, headers=headers)
    rec_tracks = recs.json()['tracks']
    for track in rec_tracks:
        artist = track['artists'][0]
        name = artist['name']
        id = artist['id']
        result = {'artist_name':name, 'artist_id':id}
        results.append(result)
    print(f'{idx+1} / {len(genre_seeds)}', end='\r')

In [30]:
genre_artists_df = pd.DataFrame(results)

In [31]:
genre_artists_df = pop_df

In [32]:
genre_artists_df = genre_artists_df.drop_duplicates().reset_index(drop=True)

In [33]:
genre_artists_df.artist_id

,artist_id
0,1scVfBymTr3CeZ4imMj1QJ
1,00x1fYSGhdqScXBRpSj3DW
2,6jJ0s89eD6GaHleKKya26X
3,53XhwfbYqKCa1cC15pYq2q
4,0TnOYISbd1XYRBk9myaseg
...,...
195,4gvjmrtzydbMpyJaXUtwvP
196,4npEfmQ6YuiwW1GpUmaq3F
197,07YZf4WDAMNwqr4jfgOZ8y
198,014WIDx7H4BRCHB1faiisK


In [34]:
chunk_size = math.ceil(len(genre_artists_df) / 50)

In [35]:
chunk_size

4

In [36]:
genre_artists_df['genres'] = float('nan')
genre_artists_df['popularity'] = float('nan')

In [37]:
genre_artists_df

,artist_name,artist_id,genres,popularity
0,Forrest Frank,1scVfBymTr3CeZ4imMj1QJ,NaN,NaN
1,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,NaN,NaN
2,Katy Perry,6jJ0s89eD6GaHleKKya26X,NaN,NaN
3,Imagine Dragons,53XhwfbYqKCa1cC15pYq2q,NaN,NaN
4,Pitbull,0TnOYISbd1XYRBk9myaseg,NaN,NaN
...,...,...,...,...
195,Addison Rae,4gvjmrtzydbMpyJaXUtwvP,NaN,NaN
196,Ava Max,4npEfmQ6YuiwW1GpUmaq3F,NaN,NaN
197,Jason Derulo,07YZf4WDAMNwqr4jfgOZ8y,NaN,NaN
198,Los Tucanes De Tijuana,014WIDx7H4BRCHB1faiisK,NaN,NaN


In [21]:
# print(several_artists.json())

NameError: name 'several_artists' is not defined

In [38]:
genre_artists_full_results = []
for artists in np.array_split(genre_artists_df, 20):
    params = {'ids' : ','.join(list(artists.artist_id))}
    several_artists = requests.get(BASE_URL+'/artists/', params=params, headers=headers)
    for i in artists.index:
        j = i - artists.index[0]
        result = {
            'artist_name': genre_artists_df.loc[i, 'artist_name'],
            'artist_id': genre_artists_df.loc[i, 'artist_id'],
            'genres': several_artists.json()['artists'][j]['genres'],
            'popularity': several_artists.json()['artists'][j]['popularity']
        }
        genre_artists_full_results.append(result)
        print(f'{i+1} / {len(genre_artists_df)}', end= '\r')

genre_artists_df = pd.DataFrame(genre_artists_full_results)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [39]:
genre_artists_df

,artist_name,artist_id,genres,popularity
0,Forrest Frank,1scVfBymTr3CeZ4imMj1QJ,"[christian, christian pop, christian hip hop, ...",78
1,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,[pop soul],89
2,Katy Perry,6jJ0s89eD6GaHleKKya26X,[pop],84
3,Imagine Dragons,53XhwfbYqKCa1cC15pYq2q,[],86
4,Pitbull,0TnOYISbd1XYRBk9myaseg,[],84
...,...,...,...,...
195,Addison Rae,4gvjmrtzydbMpyJaXUtwvP,[],74
196,Ava Max,4npEfmQ6YuiwW1GpUmaq3F,[],78
197,Jason Derulo,07YZf4WDAMNwqr4jfgOZ8y,[],77
198,Los Tucanes De Tijuana,014WIDx7H4BRCHB1faiisK,"[corrido, norteño, cumbia norteña, música mexi...",75


In [40]:
#extracting tracks from artists
def get_headers():
    return authenticate_token()  # your existing function

def fetch_tracks_for_artists(artists_df, market="US"):
    all_tracks = []
    headers = get_headers()

    for idx, row in artists_df.iterrows():
        artist_id = row["artist_id"]
        artist_name = row["artist_name"]
        artist_genres = row["genres"]
        artist_popularity = row["popularity"]

        url = f"{BASE_URL}/artists/{artist_id}/top-tracks"
        params = {"market": market}

        r = requests.get(url, headers=headers, params=params)

        # basic retry for expired token / rate limit
        while not r.ok:
            if r.status_code == 401:
                headers = get_headers()
                r = requests.get(url, headers=headers, params=params)
            elif r.status_code == 429:
                time.sleep(30)
                r = requests.get(url, headers=headers, params=params)
            else:
                break

        if not r.ok:
            print(f"Skipping {artist_name} ({artist_id}) – status {r.status_code}")
            continue

        for t in r.json().get("tracks", []):
            all_tracks.append({
                "track_id": t["id"],
                "track_name": t["name"],
                "track_preview_link": t.get("preview_url"),
                "track_popularity": t.get("popularity"),
                "track_uri": t.get("uri"),
                "artist_name": artist_name,
                "artist_id": artist_id,
                "artist_genres": artist_genres,
                "artist_popularity": artist_popularity,
                "release_date": t["album"].get("release_date"),
            })

        print(f"{idx+1} / {len(artists_df)} artists processed", end="\r")

    print()
    tracks_raw = pd.DataFrame(all_tracks).drop_duplicates(subset=["track_id"])
    return tracks_raw


tracks_raw = fetch_tracks_for_artists(genre_artists_df)


200 / 200 artists processed


In [41]:
print(tracks_raw.head())

                 track_id              track_name track_preview_link  \
0  7JDfWHxOFo63yQmVs5wSPM       YOUR WAY'S BETTER               None   
1  0vC82BouGPXm6X2K60RfQw                GOOD DAY               None   
2  1Bgj8C4oHOR5M3wuzb6Mmq                     UP!               None   
3  29UECtQE2aqbuHIjvAlAU8                LEMONADE               None   
4  25f7KnoDqO7nBbPahi15UE  NEVER GET USED TO THIS               None   

   track_popularity                             track_uri    artist_name  \
0                75  spotify:track:7JDfWHxOFo63yQmVs5wSPM  Forrest Frank   
1                75  spotify:track:0vC82BouGPXm6X2K60RfQw  Forrest Frank   
2                73  spotify:track:1Bgj8C4oHOR5M3wuzb6Mmq  Forrest Frank   
3                68  spotify:track:29UECtQE2aqbuHIjvAlAU8  Forrest Frank   
4                72  spotify:track:25f7KnoDqO7nBbPahi15UE  Forrest Frank   

                artist_id                                      artist_genres  \
0  1scVfBymTr3CeZ4imMj1QJ  [ch

In [43]:
#extracting the tracks that are released on or after 2000

# make sure release_date is string
tracks_raw["release_date"] = tracks_raw["release_date"].astype(str)

# extract year and filter
tracks_raw["release_year"] = tracks_raw["release_date"].str[:4].astype(int)
tracks_2000_plus = tracks_raw[tracks_raw["release_year"] >= 2000].copy()

# put columns in exactly the order you want
cols = [
    "track_id",
    "track_name",
    "track_preview_link",
    "track_popularity",
    "track_uri",
    "artist_name",
    "artist_id",
    "artist_genres",
    "artist_popularity",
    "release_date",
]

tracks_2000_plus = tracks_2000_plus[cols].reset_index(drop=True)

tracks_2000_plus.tail()


,track_id,track_name,track_preview_link,track_popularity,track_uri,artist_name,artist_id,artist_genres,artist_popularity,release_date
1683,24futxi7j75kgGXiumTEn2,Don't Let Me Drown - From F1® The Movie,None,64,spotify:track:24futxi7j75kgGXiumTEn2,F1 The Album,3aly4xJOy3LVznzvRIvFYC,[],76,2025-06-26
1684,2TuVErkUG3BdQR0dsbtakg,OMG! (From F1® The Movie),None,65,spotify:track:2TuVErkUG3BdQR0dsbtakg,F1 The Album,3aly4xJOy3LVznzvRIvFYC,[],76,2025-06-05
1685,5xalbHoIf0F0AmuTKlm2Ct,No Room For A Saint (From F1® The Movie),None,63,spotify:track:5xalbHoIf0F0AmuTKlm2Ct,F1 The Album,3aly4xJOy3LVznzvRIvFYC,[],76,2025-05-16
1686,7KVT3NYXWguwJ0zzup1q1S,Underdog (From F1® The Movie),None,63,spotify:track:7KVT3NYXWguwJ0zzup1q1S,F1 The Album,3aly4xJOy3LVznzvRIvFYC,[],76,2025-06-13
1687,7jtFKlVjEgIBNaSOrAivCw,DOUBLE C - From F1® The Movie,None,59,spotify:track:7jtFKlVjEgIBNaSOrAivCw,F1 The Album,3aly4xJOy3LVznzvRIvFYC,[],76,2025-06-26


In [24]:
related_dfs = [genre_artists_df]

In [25]:
for i in range(1,3):
    new_artists = []
    for idx, artist in related_dfs[i-1].iterrows():
        related = requests.get(BASE_URL+'/artists/'+artist.artist_id+'/related-artists', headers=headers)
        while(related.ok == False):
            if related.status_code == 401:
                headers = authenticate_token()
                related = requests.get(BASE_URL+'/artists/'+artist.artist_id+'/related-artists', headers=headers)
            elif related.status_code == 429:
                time.sleep(30)
                related = requests.get(BASE_URL+'/artists/'+artist.artist_id+'/related-artists', headers=headers)
            else:
                break
        for new_artist in related.json()['artists']:
            new_row = {'artist_name': new_artist['name'],
                      'artist_id': new_artist['id'],
                      'genres': new_artist['genres'],
                      'popularity': new_artist['popularity']}
            new_artists.append(new_row)
        print(f'{idx+1} / {len(related_dfs[i-1])}', end='\r')
    print('\n')
    related_dfs.append(pd.DataFrame(new_artists))
    related_dfs[i] = related_dfs[i].drop_duplicates(subset=['artist_id'])


KeyError: 'artists'

In [26]:
all_artists = pd.concat([df for df in related_dfs])

In [27]:
all_artists = all_artists.drop_duplicates(subset=['artist_id']).reset_index(drop=True)

In [28]:
all_tracks = []

In [31]:
for idx, artist in all_artists.iterrows():
    top_tracks = requests.get(BASE_URL+'/artists/'+artist.artist_id+'/top-tracks?market=US', headers=headers)
    for track in top_tracks.json()['tracks']:
        track_info = {
            'track_id': track['id'],
            'track_name': track['name'],
            'track_preview_link': track['preview_url'],
            'track_popularity': track['popularity'],
            'track_uri': track['uri'],
            'release_date':track['album']['release_date'],
            'artist_name': artist.artist_name,
            'artist_id': artist.artist_id,
            'artist_genres': artist.genres,
            'artist_popularity': artist.popularity
        }
        all_tracks.append(track_info)
    print(f'{idx+1} / {len(all_artists)}', end='\r')
all_tracks_df = pd.DataFrame(all_tracks)

In [32]:
track_chunk_size

NameError: name 'track_chunk_size' is not defined

In [ ]:
# get the release date of the tracks, then continue to run and get this file is done, then move to scrap reviews

In [42]:
track_chunk_size = math.ceil(len(all_tracks) / 50)

release_dates = []
for tracks in np.array_split(all_tracks, track_chunk_size):
  # print(tracks)
  track_ids = [x["track_id"] for x in tracks]
  params = {'ids' : ','.join(track_ids),
             'market': 'US'}
  several_tracks = requests.get(BASE_URL+'/tracks/', params=params, headers=headers)
  for i in tracks.index:
      j = i - tracks.index[0]
      result = {
          'track_id': several_tracks.json()['tracks'][j]['id'],
          'release_date': several_tracks.json()['tracks'][j]['album']['release_date']
      }
      release_dates.append(result)
      print(f'{i+1} / {len(all_tracks)}', end= '\r')

AttributeError: 'numpy.ndarray' object has no attribute 'index'

In [ ]:
len(release_dates)

459111

In [ ]:
from pyarrow import feather
feather.write_feather(all_tracks, 'data/all_tracks.feather')
feather.write_feather(all_artists, 'data/all_artists.feather')